# e001 training

Trains YOLO26s on rdd5-v1 and evaluates on the test split. Runs on Colab (T4). Results in `ml/EXPERIMENTS.md`.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!git clone https://github.com/nataliovcharov/dupka.git /content/dupka
%cd /content/dupka/ml
!git log --oneline -1

In [ ]:
!pip install -q "ultralytics==8.4.165" remotezip

In [ ]:
%cd /content/dupka/ml
!mkdir -p data/raw
%cd data/raw
!remotezip "https://ndownloader.figshare.com/files/38030910" RDD2022/Czech.zip RDD2022/Japan.zip RDD2022/India.zip RDD2022/United_States.zip RDD2022/China_MotorBike.zip
!ls -lh RDD2022

In [ ]:
!cd RDD2022 && for f in *.zip; do unzip -q "$f"; done && ls RDD2022 2>/dev/null || ls RDD2022
%cd /content/dupka/ml
!ls data/raw/RDD2022

In [ ]:
!python scripts/convert_rdd_to_yolo.py --src data/raw/RDD2022 --out data/yolo/rdd5-v1 --countries Czech Japan India United_States China_MotorBike
!cat data/yolo/rdd5-v1/data.yaml

In [ ]:
!python scripts/visualize_labels.py --data data/yolo/rdd5-v1 --n 6
import glob

from IPython.display import Image, display

for path in sorted(glob.glob("data/viz/*.jpg")):
    display(Image(path, width=400))

In [ ]:
from ultralytics import YOLO

RUN = "e001-yolo26s-rdd5-baseline"
PROJECT = "/content/drive/MyDrive/dupka/runs"

model = YOLO("yolo26s.pt")
model.train(
    data="data/yolo/rdd5-v1/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    workers=2,
    seed=42,
    patience=10,
    project=PROJECT,
    name=RUN,
    exist_ok=True,
)

In [ ]:
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/dupka/runs/e001-yolo26s-rdd5-baseline/weights/best.pt"
)
metrics = model.val(
    data="data/yolo/rdd5-v1/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device="cpu",
    project="/content/drive/MyDrive/dupka/runs",
    name="e001-yolo26s-rdd5-baseline-test",
    exist_ok=True,
)